# StateSet Agents v1.0 — Interactive 3D Weight Studio, Live RL Cockpit & MCP Control Plane (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/weight_studio_3d_colab.ipynb)

> **Runs on CPU or GPU in Google Colab — no ngrok or external tunnels required!**
>
> What you will explore interactively inside Colab:
> 1. Launch **3D Weight Studio (`stateset-agents studio --port 8765`)** embedded directly inside your notebook via Colab's native port proxy.
> 2. Trigger the **6-Domain Enterprise Verifiable Sandbox Pool** (`commerce`, `coding`, `sql`, `rest_api`, `docker_code`, `web_agent`) and inspect per-turn PRM credit in real time.
> 3. Run **Autonomous Self-Heal → 10-Probe Red-Team Immunization → Paired Shadow Canary Gate** and watch 3D SSE events animate live.
> 4. Stream live **`StudioTelemetryCallback`** RL steps with bi-directional Pause/Rollback control and query the **Streamable HTTP MCP** endpoint.
> 5. Download a standalone, zero-dependency **Interactive 3D HTML PR Report**.

## 1. Install `stateset-agents[api,mcp]`

In [ ]:
import os
import subprocess
import sys

INSTALL_SPEC = os.environ.get(
    "STATESET_STUDIO_INSTALL_SPEC",
    "stateset-agents[api,mcp] @ git+https://github.com/stateset/stateset-agents.git@release/0.99.1",
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", INSTALL_SPEC])

import stateset_agents
print("Installed stateset-agents:", stateset_agents.__version__)

## 2. Start 3D Weight Studio & Embed Live WebGL2 UI in Colab

In [ ]:
import subprocess
import sys
import time
import urllib.request

from stateset_agents.utils.colab import is_colab_runtime

PORT = 8765
studio_proc = subprocess.Popen(
    [sys.executable, "-m", "stateset_agents.weight_studio", "--host", "127.0.0.1", "--port", str(PORT), "--workspace", "."],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/weight-studio/health", timeout=1.0) as r:
            if r.status == 200:
                print("3D Weight Studio healthy on port", PORT)
                break
    except Exception:
        time.sleep(0.25)

if is_colab_runtime():
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, path="/studio", height=760)
    print("Tip: You can also open the Studio in a full browser tab using:")
    output.serve_kernel_port_as_window(PORT, path="/studio")
else:
    print(f"Open http://127.0.0.1:{PORT}/studio in your browser.")

## 3. Drive the Live 3D RL Cockpit, 6-Domain Sandbox Pool & Streamable HTTP MCP

In [ ]:
import json
import urllib.request

def post_json(path: str, payload: dict) -> dict:
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode("utf-8"))

# 1. Run 6-Domain Enterprise Verifiable Sandbox Pool (animates live in 3D Studio!)
pool = post_json(
    "/api/weight-studio/rl/sandbox-pool",
    {
        "domains": ["commerce", "coding", "sql", "rest_api", "docker_code", "web_agent"],
        "include_failure_pairs": True,
    },
)
print("Sandbox Pool episodes:", pool.get("episode_count"), "domains:", sorted((pool.get("domain_stats") or {}).keys()))

# 2. Stream live RL telemetry steps into SQLite WAL ledger & sparkline HUD
for step in range(1, 5):
    post_json(
        "/api/weight-studio/rl/telemetry",
        {
            "run_name": "colab-ultra-gspo",
            "step": step,
            "mean_reward": round(0.45 + step * 0.12, 3),
            "entropy": round(1.2 - step * 0.08, 3),
            "kl": 0.018,
            "guard_verdict": "HEALTHY",
        },
    )

# 3. Call Streamable HTTP MCP directly (32 canonical Studio tools)
mcp_res = post_json(
    "/api/weight-studio/mcp",
    {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}},
)
tools = (mcp_res.get("result") or {}).get("tools") or []
print(f"Streamable HTTP MCP exposed {len(tools)} annotated tools.")

## 4. Run Autonomous Autopilot (Auto-Heal → Red-Team → Canary) & Download Standalone 3D HTML Report

In [ ]:
from pathlib import Path
from stateset_agents.utils.colab import is_colab_runtime

autopilot = post_json(
    "/api/weight-studio/autopilot/run",
    {"model_id": "preset:deepseek-r1-1.5b", "reward_fn": "customer_support"},
)
print("Autopilot Canary Decision:", (autopilot.get("canary") or {}).get("decision"))

html_res = post_json(
    "/api/weight-studio/report/html",
    {"model_id": "preset:deepseek-r1-1.5b", "title": "StateSet v1.0 Colab 3D Surgery Report"},
)
report_path = Path("outputs/colab_studio_pr_report.html")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(html_res.get("html", ""), encoding="utf-8")
print("Saved standalone 3D HTML report:", report_path, f"({report_path.stat().st_size} bytes)")

if is_colab_runtime():
    from google.colab import files
    files.download(str(report_path))